# Zadanie 1

**Budowa pipeline ETL dla danych sprzedaży**

Extract – wczytanie danych

Transform – czyszczenie i przygotowanie

Load – zapis danych

# Extract

In [126]:
import pandas as pd

df = pd.read_csv("Online_Retail.csv", encoding="latin1")

df.head()
df.info()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


# Transform

**Usunięcie brakujących CustomerID**

In [127]:
df = df.dropna(subset=["CustomerID"])

**Usunięcie złych wartości**

Quantity <= 0

In [128]:
df = df[df["Quantity"] > 0]

UnitPrice < 0

In [129]:
df = df[df["UnitPrice"] >= 0]

**Usunięcie duplikatów**

In [130]:
df = df.drop_duplicates()

**Poprawienie typu danych**

In [131]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype(int)

/tmp/ipykernel_5667/2496147954.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


**Rozbicie daty**

In [132]:
df = df.dropna(subset=["InvoiceDate"])

In [133]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [134]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day

**Sprawdzenie po czyszczeniu**

In [135]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,2010,12,1
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,12,1
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,2010,12,1
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,12,1
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2010,12,1


**Jak przygotować dane, aby łatwo analizować wartość sprzedaży i przychód?**

Dane należy przygotować poprzez dodanie nowej kolumny Revenue, obliczonej jako iloczyn liczby produktów (Quantity) oraz ceny jednostkowej (UnitPrice). Dzięki temu możliwa jest bezpośrednia analiza wartości sprzedaży.

**Czy potrzebne są dodatkowe kolumny?**

Tak, konieczne jest dodanie kolumny Revenue, która reprezentuje wartość transakcji. Dodatkowo można rozważyć utworzenie kolumny łączącej rok i miesiąc (np. YearMonth) w celu analizy czasowej.

**Czy warto coś policzyć już na etapie ETL?**

Tak, warto obliczyć podstawowe miary, takie jak przychód (Revenue), już na etapie ETL, ponieważ:

upraszcza to późniejszą analizę danych

eliminuje konieczność wielokrotnego liczenia tych samych wartości

poprawia czytelność i użyteczność zbioru danych

**Utworzenie tabeli faktów**

In [136]:
fact_table = df[[
    "InvoiceNo",
    "StockCode",
    "CustomerID",
    "InvoiceDate",
    "Quantity",
]]

# Load

**Eksport tabeli faktów do pliku CSV**

In [137]:
fact_table.to_csv("fact_sales.csv", index=False)

# Zadanie 2

**Wczytanie zbiorów danych**

In [138]:
df1 = pd.read_csv("Online_Retail.csv", encoding="latin1")
df2 = pd.read_excel("online_retail_II.xlsx")

**Porównanie zbiorów**

In [139]:
df1.columns
df2.columns

df1.dtypes
df2.dtypes

df1.isnull().sum()
df2.isnull().sum()

,0
Invoice,0
StockCode,0
Description,2928
Quantity,0
InvoiceDate,0
Price,0
Customer ID,107927
Country,0


**Czy struktura danych jest identyczna?**

Nie, struktura danych nie jest identyczna. Występują różnice w nazwach kolumn (np. CustomerID vs Customer ID, InvoiceNo vs Invoice, UnitPrice vs Price) oraz w liczbie brakujących wartości.

**Czy dane można od razu połączyć?**

Nie, danych nie można od razu połączyć. Przed połączeniem należy:

ujednolicić nazwy kolumn

upewnić się, że typy danych są zgodne

obsłużyć brakujące wartości

**Ujednolicenie nazw kolumn**

In [140]:
df2 = df2.rename(columns={
    "Invoice": "InvoiceNo",
    "Customer ID": "CustomerID",
    "Price": "UnitPrice"
})

**Ujednolicenie typów danych**

In [141]:
df1["CustomerID"] = df1["CustomerID"].astype("Int64")
df2["CustomerID"] = df2["CustomerID"].astype("Int64")

**Ujednolicenie dat**

In [142]:
df1["InvoiceDate"] = pd.to_datetime(df1["InvoiceDate"])
df2["InvoiceDate"] = pd.to_datetime(df2["InvoiceDate"])

/tmp/ipykernel_5667/670812180.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df1["InvoiceDate"] = pd.to_datetime(df1["InvoiceDate"])


**Połączenie danych**

In [143]:
df = pd.concat([df1, df2], ignore_index=True)

**Który schemat przyjąć jako docelowy?**

Jako schemat docelowy należy przyjąć strukturę pierwszego zbioru danych (df1), ponieważ zawiera spójne nazwy kolumn bez spacji oraz bardziej standardowe nazewnictwo.

**Czy wszystkie kolumny są potrzebne?**

Nie, nie wszystkie kolumny są konieczne. Do analizy sprzedaży wystarczą kluczowe kolumny, takie jak:

InvoiceNo

StockCode

CustomerID

InvoiceDate

Quantity

UnitPrice

Pozostałe kolumny mogą być opcjonalne w zależności od analizy.

# Data Quality

**Wykrywanie duplikatów**

In [144]:
duplicates = df[df.duplicated(subset=[
    "InvoiceNo", "StockCode", "CustomerID", "InvoiceDate"
])]

**Wykrywanie niespójności**

In [145]:
df.groupby("StockCode")["UnitPrice"].nunique().sort_values(ascending=False)

,UnitPrice
StockCode,
DOT,1291
M,513
POST,180
D,162
S,94
...,...
DCGS0073,1
DCGS0071,1
16010,1


**Jak rozpoznać duplikat?**

Duplikaty można rozpoznać na podstawie identycznych wartości w kluczowych kolumnach, takich jak InvoiceNo, StockCode, CustomerID oraz InvoiceDate.

**Co zrobić z konfliktem danych?**

W przypadku konfliktów danych należy:

wybrać jedną wartość

lub usunąć niejednoznaczne rekordy

**Które źródło jest bardziej wiarygodne?**

Bardziej wiarygodne jest źródło, które:

zawiera mniej brakujących wartości

ma spójniejsze dane

jest bardziej aktualne i kompletne

# Merge

In [146]:
df_all = pd.concat([df1, df2], ignore_index=True)

**Czy użyć concat czy merge?**

Należy użyć funkcji concat, ponieważ oba zbiory danych mają tę samą strukturę i chcemy je połączyć jeden pod drugim.

Funkcja merge służy do łączenia danych na podstawie wspólnych kolumn, co w tym przypadku nie jest potrzebne.

**Czy zachować wszystkie rekordy?**

Tak, należy zachować wszystkie rekordy, aby nie utracić żadnych informacji.
Ewentualne duplikaty lub błędne dane można usunąć na etapie czyszczenia danych.

# Load

**Zapis zintegrowanych danych**

In [147]:
fact_table.to_csv("fact_sales_integrated.csv", index=False)